# Phase 2 Data Pipeline Validation

This notebook validates the reproducible Phase 2 data/metrics pipeline for the qBraid / JonesTrading Track A volatility-regime project.

Scope for this notebook:

- load the processed SPY+VIX fallback dataset;
- verify chronological train/validation/test splits;
- verify train-only transition-threshold construction;
- build leakage-safe normalized regression arrays;
- build sequence arrays for ESN/QRC-style models;
- evaluate the persistence baseline floor using Track A metrics.

This is a data-pipeline validation notebook, not a model-optimization notebook. Classical baseline modeling is handled separately.

## 1. Imports

In [15]:
import pandas as pd

from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.splits import (
    add_train_only_transition_flags,
    chronological_tabular_split,
    describe_regression_splits,
)
from qpitome_qrc.data.features import (
    FEATURE_COLUMNS,
    make_regression_arrays,
    make_sequence_arrays,
)
from qpitome_qrc.evaluation.metrics import (
    evaluate_volatility_forecast,
    volatility_metrics_to_frame,
)

## 2. Load processed fallback dataset

The executable Phase 2 path uses the processed SPY+VIX fallback dataset. VOLARE can remain a preferred scientific upgrade, but this path is immediately reproducible without gated data access.

In [4]:
from pathlib import Path

import os

if Path.cwd().name == "notebooks":

    os.chdir("..")

df = load_phase2_volatility_data()

print(f"shape: {df.shape}")
print(f"date range: {df['date'].min()} -> {df['date'].max()}")
df[["future_rv_5d", "future_rv_20d"]].describe(percentiles=[0.25, 0.5, 0.75])

shape: (7775, 41)
date range: 1993-04-27 00:00:00 -> 2024-03-15 00:00:00


,future_rv_5d,future_rv_20d
count,7775.000000,7775.000000
mean,0.151704,0.159231
std,0.110807,0.099694
min,0.009001,0.032771
25%,0.082241,0.096241
50%,0.124261,0.134265
75%,0.188900,0.194653
max,1.342073,0.924247


## 3. Validate chronological splits

All splits are chronological. No random shuffling is used for this time-series forecasting task.

In [6]:
splits = chronological_tabular_split(df)
split_summary = describe_regression_splits(
    splits,
    targets=["future_rv_5d", "future_rv_20d"],
)
split_summary

,split,n_rows,date_min,date_max,future_rv_5d_mean,future_rv_5d_std,future_rv_5d_q50,future_rv_5d_q80,future_rv_20d_mean,future_rv_20d_std,future_rv_20d_q50,future_rv_20d_q80
0,train,5459,1993-04-27,2014-12-31,0.156068,0.110057,0.129531,0.210931,0.163352,0.099046,0.139874,0.212559
1,val,1258,2015-01-02,2019-12-31,0.112417,0.073590,0.093502,0.160788,0.119594,0.060372,0.100049,0.170999
2,test,1058,2020-01-02,2024-03-15,0.175900,0.137100,0.145220,0.228209,0.185094,0.124425,0.154025,0.236229


## 4. Validate train-only transition thresholds

Transition flags are used for interpretation and regime-warning analysis. Thresholds are computed from training data only, then applied unchanged to validation/test data.

In [8]:
df_flags, transition_info = add_train_only_transition_flags(df)

transition_rows = []
for split_name, split_df in chronological_tabular_split(df_flags).items():
    transition_rows.append(
        {
            "split": split_name,
            "count": len(split_df),
            "events": int(split_df["transition_event"].sum()),
            "event_rate": float(split_df["transition_event"].mean()),
        }
    )

transition_counts = pd.DataFrame(transition_rows)

transition_info, transition_counts

({'calm_threshold': 0.1389738004829988,
  'turbulent_threshold': 0.2125594115019885,
  'calm_quantile': 0.5,
  'turbulent_quantile': 0.8},
    split  count  events  event_rate
 0  train   5459      88    0.016120
 1    val   1258      57    0.045310
 2   test   1058      28    0.026465)

## 5. Build leakage-safe normalized regression arrays

Feature normalization is performed after splitting. The scaler is fit on the training split only and applied unchanged to validation and test splits.

In [9]:
arrays_20d, scaler_20d = make_regression_arrays(
    splits,
    target_column="future_rv_20d",
    scaler_name="standard",
)

array_summary = pd.DataFrame(
    [
        {"split": "train", "X_shape": arrays_20d.X_train.shape, "y_shape": arrays_20d.y_train.shape},
        {"split": "validation", "X_shape": arrays_20d.X_val.shape, "y_shape": arrays_20d.y_val.shape},
        {"split": "test", "X_shape": arrays_20d.X_test.shape, "y_shape": arrays_20d.y_test.shape},
    ]
)

print(f"feature_count: {len(arrays_20d.feature_columns)}")
print(f"target: {arrays_20d.target_column}")
print(f"scaler: {arrays_20d.scaler_name}")
array_summary

feature_count: 27
target: future_rv_20d
scaler: standard


,split,X_shape,y_shape
0,train,"(5459, 27)","(5459,)"
1,validation,"(1258, 27)","(1258,)"
2,test,"(1058, 27)","(1058,)"


## 6. Build sequence arrays for reservoir models

A 20-trading-day lookback provides model-ready sequence arrays for ESN/QRC-style reservoirs. The target remains the forward-looking realized-volatility target already constructed in the processed dataset.

In [10]:
X_seq, y_seq, seq_dates = make_sequence_arrays(
    splits["train"],
    target_column="future_rv_20d",
    lookback=20,
)

print(f"X_train_sequence: {X_seq.shape}")
print(f"y_train_sequence: {y_seq.shape}")
print(f"first_sequence_target_date: {seq_dates.iloc[0]}")
print(f"last_sequence_target_date: {seq_dates.iloc[-1]}")
print(f"sequence_count_check: {len(splits['train'])} - 20 + 1 = {len(splits['train']) - 20 + 1}")

X_train_sequence: (5440, 20, 27)
y_train_sequence: (5440,)
first_sequence_target_date: 1993-05-24 00:00:00
last_sequence_target_date: 2014-12-31 00:00:00
sequence_count_check: 5459 - 20 + 1 = 5440


## 7. Evaluate persistence baseline floor

Persistence forecasts are the first baseline floor: current trailing realized volatility is used as the forecast for future realized volatility.

In [12]:
baseline_specs = [
    ("persistence_5d_to_5d", "future_rv_5d", "rv_5d"),
    ("persistence_10d_to_5d", "future_rv_5d", "rv_10d"),
    ("persistence_20d_to_20d", "future_rv_20d", "rv_20d"),
    ("persistence_60d_to_20d", "future_rv_20d", "rv_60d"),
]

rows = []
for split_name, split_df in splits.items():
    for model_name, target_col, predictor_col in baseline_specs:
        metrics = evaluate_volatility_forecast(
            split_df[target_col].to_numpy(),
            split_df[predictor_col].to_numpy(),
        )

        row = {
            "rmse": metrics.rmse,
            "qlike": metrics.qlike,
            "mz_alpha": metrics.mz_alpha,
            "mz_beta": metrics.mz_beta,
            "mz_r2": metrics.mz_r2,
        }
        row.update(
            {
                "split": split_name,
                "model": model_name,
                "target": target_col,
                "predictor": predictor_col,
                "n": len(split_df),
            }
        )
        rows.append(row)

baseline_table = pd.DataFrame(rows)
baseline_table = baseline_table[
    ["split", "model", "target", "predictor", "n", "rmse", "qlike", "mz_alpha", "mz_beta", "mz_r2"]
]
baseline_table.sort_values(["target", "split", "rmse"])


,split,model,target,predictor,n,rmse,qlike,mz_alpha,mz_beta,mz_r2
10,test,persistence_20d_to_20d,future_rv_20d,rv_20d,1058,0.124308,-2.004007,0.092637,0.500470,0.251561
11,test,persistence_60d_to_20d,future_rv_20d,rv_60d,1058,0.134645,-1.661863,0.108992,0.395942,0.127195
2,train,persistence_20d_to_20d,future_rv_20d,rv_20d,5459,0.073516,-2.547335,0.045226,0.724148,0.525179
3,train,persistence_60d_to_20d,future_rv_20d,rv_60d,5459,0.077340,-2.600812,0.041370,0.728625,0.454956
7,val,persistence_60d_to_20d,future_rv_20d,rv_60d,1258,0.064837,-2.886382,0.070186,0.389747,0.096572
6,val,persistence_20d_to_20d,future_rv_20d,rv_20d,1258,0.065048,-2.759242,0.068901,0.419395,0.176100
9,test,persistence_10d_to_5d,future_rv_5d,rv_10d,1058,0.102459,-2.221573,0.041439,0.743875,0.501707
8,test,persistence_5d_to_5d,future_rv_5d,rv_5d,1058,0.102561,-1.888258,0.049263,0.719934,0.518291
1,train,persistence_10d_to_5d,future_rv_5d,rv_10d,5459,0.078689,-2.622180,0.031650,0.776063,0.534719
0,train,persistence_5d_to_5d,future_rv_5d,rv_5d,5459,0.086109,-2.119481,0.047807,0.693864,0.481455


## 8. May 22 sign-off summary

In [13]:
signoff = {
    "processed_rows": len(df),
    "processed_columns": df.shape[1],
    "feature_count": len(FEATURE_COLUMNS),
    "targets": ["future_rv_5d", "future_rv_20d"],
    "train_rows": len(splits["train"]),
    "validation_rows": len(splits["val"]),
    "test_rows": len(splits["test"]),
    "tabular_train_shape": arrays_20d.X_train.shape,
    "sequence_train_shape": X_seq.shape,
    "track_a_metrics": ["RMSE", "QLIKE", "Mincer-Zarnowitz"],
}
signoff

{'processed_rows': 7775,
 'processed_columns': 41,
 'feature_count': 27,
 'targets': ['future_rv_5d', 'future_rv_20d'],
 'train_rows': 5459,
 'validation_rows': 1258,
 'test_rows': 1058,
 'tabular_train_shape': (5459, 27),
 'sequence_train_shape': (5440, 20, 27),
 'track_a_metrics': ['RMSE', 'QLIKE', 'Mincer-Zarnowitz']}